## Interpreting pretrained TTM with WinTSR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/pretrained_ttm.ipynb)

Use IBM's tiny [TTM](https://arxiv.org/abs/2401.03955) forecasting model to make a real zero-shot forecast and reveal which observations drove it with WinTSR. The [Granite TTM-R2 checkpoint](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2) has only a few million parameters, making this the most Colab-friendly tutorial in the series.

In [ ]:
%pip install -q tslens granite-tsfm

## 1. Load TTM and make a zero-shot forecast

We give the pretrained model 512 hourly oil-temperature observations from ETTh2 and ask for the next 96, with no training or fine-tuning.

In [ ]:
import pandas as pd
import torch
from tsfm_public.toolkit.get_model import get_model

DATA_URL = "https://raw.githubusercontent.com/WenWeiTHU/TimeSeriesDatasets/refs/heads/main/ETT-small/ETTh2.csv"
CONTEXT_LEN, PREDICTION_LEN = 512, 96
BATCH_SIZE = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

series = torch.tensor(pd.read_csv(DATA_URL)["OT"].dropna().to_numpy(), dtype=torch.float32)
starts = tuple(24 * i for i in range(BATCH_SIZE))
past_values = torch.stack([series[s : s + CONTEXT_LEN] for s in starts]).unsqueeze(-1).to(device)
ground_truth = series[CONTEXT_LEN : CONTEXT_LEN + PREDICTION_LEN]

model = get_model(
    model_path="ibm-granite/granite-timeseries-ttm-r2",
    context_length=512,
    prediction_length=96,
)
model.to(device).eval()
with torch.inference_mode():
    output = model(past_values=past_values[:1])
prediction = output.prediction_outputs[0, :, 0].detach().cpu()
print(f"device: {device} | input: {tuple(past_values[:1].shape)} | forecast: {tuple(output.prediction_outputs.shape)}")

In [ ]:
import matplotlib.pyplot as plt

context = past_values[0, :, 0].detach().cpu()
forecast_steps = torch.arange(CONTEXT_LEN, CONTEXT_LEN + PREDICTION_LEN)
plt.figure(figsize=(12, 4))
plt.plot(torch.arange(CONTEXT_LEN), context, label="lookback", linewidth=1)
plt.plot(forecast_steps, ground_truth, label="ground truth", linewidth=2)
plt.plot(forecast_steps, prediction, label="TTM forecast", linewidth=2)
plt.axvline(CONTEXT_LEN - 1, color="0.4", linestyle="--", linewidth=1)
plt.xlabel("hour")
plt.ylabel("oil temperature (OT)")
plt.title("TTM zero-shot forecast on ETTh2")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 2. The ideal WinTSR interface

Unlike Timer or MOMENT, TTM's own calling convention already matches tslens's -- this is the ideal case. Both use `(batch, seq_len, n_features)` tensors, so the wrapper only names TTM's input and extracts its forecast.

In [ ]:
class TTMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(past_values=x).prediction_outputs

wrapped = TTMWrapper(model).eval()

## 3. Attribute eight forecasts

TTM is small enough to explain a batch of eight with the full 512-step context. We keep WinTSR's default one-step, one-feature windows; `threshold=0.5` skips the lower half of time steps only during stage two.

In [ ]:
from tslens import WinTSR

baselines = torch.zeros_like(past_values)
attr = WinTSR(wrapped).attribute(
    inputs=past_values,
    baselines=baselines,
    threshold=0.5,
    show_progress=True,
)
print("attributions:", tuple(attr.shape), "= (batch, horizon, time, feature)")

## 4. See which time steps mattered

The input and attribution share an x-axis, matching the Timer and MOMENT tutorials. We average absolute attribution over all 96 forecast horizons for sample 0.

In [ ]:
recent = past_values[0, :, 0].detach().cpu()
saliency = attr[0].abs().mean(dim=0).squeeze(-1).detach().cpu()
steps = torch.arange(-CONTEXT_LEN, 0)

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True, height_ratios=(2, 1))
axes[0].plot(steps, recent, color="tab:blue", linewidth=1)
axes[0].set_ylabel("OT")
axes[0].set_title("TTM input and WinTSR attribution")
axes[0].grid(alpha=0.25)
axes[1].fill_between(steps, saliency, color="tab:orange", alpha=0.8)
axes[1].plot(steps, saliency, color="tab:orange", linewidth=0.8)
axes[1].set_xlabel("hours before forecast")
axes[1].set_ylabel("attribution")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 5. See what `threshold` changes

The threshold is a time-relevance quantile. `0.0` retains nearly every time step in stage two; higher values produce a faster, sparser explanation. We reuse sample 0 from the main run and compute only the two endpoints.

In [ ]:
attrs_by_threshold = {0.5: attr[:1]}
for threshold in (0.0, 0.8):
    attrs_by_threshold[threshold] = WinTSR(wrapped).attribute(
        inputs=past_values[:1],
        baselines=baselines[:1],
        threshold=threshold,
        show_progress=True,
    )

maps = {t: a[0].abs().mean(dim=0).squeeze(-1).detach().cpu() for t, a in attrs_by_threshold.items()}
vmax = max(m.max().item() for m in maps.values())
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2), sharex=True)
for ax, threshold in zip(axes, (0.0, 0.5, 0.8)):
    ax.imshow(
        maps[threshold].unsqueeze(0), aspect="auto", cmap="magma",
        vmin=0, vmax=vmax, extent=(-CONTEXT_LEN, 0, 0, 1),
    )
    ax.set_title(f"threshold = {threshold}")
    ax.set_xlabel("hours before forecast")
    ax.set_yticks([])
plt.tight_layout()
plt.show()

A higher threshold should visibly remove more low-relevance locations, but it does not change stage one's ranking. Compare recurring bright lags across maps rather than over-interpreting small magnitude differences.

## Next steps

- Try a seasonal or local-mean baseline and check whether the important lags persist.
- Index `attr[:, h]` instead of averaging to explain one forecast horizon.
- Load `HUFL`, `HULL`, and `OT` together to compare time and feature relevance in a multivariate forecast.

If this was useful, please star the [repository](https://github.com/khairulislam/tslens). Please cite the following if you use our work:

```bibtex
@article{ekambaram2024ttm,
  title={Tiny Time Mixers (TTMs): Fast Pre-trained Models for Enhanced Zero/Few-Shot Forecasting of Multivariate Time Series},
  author={Ekambaram, Vijay and Jati, Arindam and Dayama, Pankaj and Mukherjee, Sumanta and Nguyen, Nam H. and Gifford, Wesley M. and Reddy, Chandra and Kalagnanam, Jayant},
  journal={arXiv preprint arXiv:2401.03955},
  year={2024}
}

@software{islam_2026_22088943,
  author       = {Islam, Md Khairul},
  title        = {tslens: A PyTorch Framework for Interpreting Time Series Deep Learning Models},
  month        = aug,
  year         = 2026,
  publisher    = {Zenodo},
  version      = {v1.0.0},
  doi          = {10.5281/zenodo.22088943},
  url          = {https://doi.org/10.5281/zenodo.22088943}
}
```